In [ ]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import random_split
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_dataset
from core.data.dataset import create_dataloaders
from core.model.bert import BertForMaskedModeling
from core.training.sampler import create_kde_sampler
from core.training.pretrainer import setup_training
from core.logger import calculate_normalization_stats, print_data_summary


In [ ]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BERT Configuration")

In [ ]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['data']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_ratings = load_dataset(
    DATASET_PATH, 
    max_seq_len=config['data']['max_seq_len']
)

print_data_summary(all_beatmaps_data)

In [ ]:
val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

print(f"Data split: {len(train_data)} training, {len(val_data)} validation")

train_data_list = [train_data.dataset[i] for i in train_data.indices]
val_data_list = [val_data.dataset[i] for i in val_data.indices]

train_difficulty_ratings = difficulty_ratings[train_data.indices]

sampler = create_kde_sampler(
    train_difficulty_ratings,
    bandwidth=config['training']['sampling']['kde_bandwidth'],
    expand_for_augmentation=config['training']['sampling']['expand_for_augmentation'],
    num_bins=config['training']['sampling'].get('num_bins', 100),
)

normalizer = calculate_normalization_stats(
    train_data_list,
    include_augmentation=config['training']['sampling']['expand_for_augmentation']
)

vector_stats = normalizer.get_vector_stats()
meta_stats = normalizer.get_metadata_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")
print(f"Metadata normalization stats for {len(meta_stats)} fields")

In [ ]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['training']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}, meta={sample_batch[2].shape}")

In [ ]:
model = BertForMaskedModeling.from_config(config, device)
model.print_info()

print("\nRunning a test forward pass with mixed precision (autocast)...")
with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        sample_vectors, sample_mask, sample_metadata = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)
        sample_metadata = sample_metadata.to(device)

        predictions, targets, _ = model(sample_vectors, sample_metadata, sample_mask)

print("\nBERT model created and tested successfully!")

In [ ]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1 
        print(f"Loaded checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting training from scratch")

print(f"Training setup complete. Starting from epoch {start_epoch + 1}")
print(f"Total epochs: {config['training']['num_epochs']}")

In [ ]:
print("\nStarting BERT training...")
print(f"BERT Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")
print(f"Components: RoPE={config['components'].get('use_rope', False)}")

if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True):
    effective_train_size = len(train_data) * 4 if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True) else len(train_data)
    print(f"Training samples: {len(train_data)} base maps -> {effective_train_size} with augmentation")
else:
    print(f"Training samples: {len(train_data)} base maps without augmentation")

metrics_tracker = trainer.train(start_epoch)

print("\nBERT training completed!")